# Sign Language Recognition — Viva Version

## Objective
- Recognize 41 static sign-language classes.
- Use deep learning and transfer learning for image classification.
- Evaluate the model and prepare it for real-time deployment.


## Project Summary
- 41 classes × 450 images = 18,450 images.
- MobileNetV2 with ImageNet weights.
- Input: 224 × 224 RGB.
- Augmentation: rotation, zoom, translation and contrast.
- Training: transfer learning + fine-tuning.
- Evaluation: accuracy, precision, recall, F1-score and confusion matrix.
- Deployment tools: OpenCV and MediaPipe.


## Tools Used
- **Python** — programming.
- **TensorFlow/Keras** — deep learning and training.
- **MobileNetV2** — pretrained CNN.
- **Google Colab** — training.
- **OpenCV** — image/webcam processing.
- **MediaPipe** — hand detection and landmarks.
- **scikit-learn** — evaluation.
- **JSON** — class mapping.


## Further Improvements
- Collect more real webcam images.
- Add different backgrounds, lighting and camera distances.
- Add more samples for M/N/S/T.
- Use MediaPipe for consistent hand localization/cropping.
- Compare with a custom CNN using stride = 1.
- Add an Unknown/Uncertain decision.
- Use LSTM, GRU or Transformer for dynamic gestures.
- Compare EfficientNet or MobileNetV3.
- Calibrate confidence and analyze errors class-by-class.


## Key Viva Conclusion
- DL achieved strong offline image classification.
- Real-time webcam input introduced a domain gap.
- MediaPipe + ML worked better because it focuses on hand geometry.
- **Main lesson: the right feature representation can matter more than model complexity.**
- Future direction: **MediaPipe + Deep Learning hybrid system**.


# Sign Language Recognition — Deep Learning Project

## Project objective

Build a 41-class sign-language image classifier using transfer learning with **MobileNetV2**, fine-tuning, data augmentation, and evaluation on a held-out test set.

The dataset contains **41 sign classes**, with **450 images per class**, giving **18,450 images in total**. The class names and ordering are preserved from the project dataset. fileciteturn47file0L224-L270

### Class mapping

The model uses the following ordering:

`1, 10, 2, 3, 4, 5, 6, 7, 8, 9, A, B, C, D, E, F, G, H, HELLO, I, I_LOVE_YOU, J, K, L, M, N, NO, O, P, Q, R, S, T, THANK_YOU, U, V, W, X, Y, YES, Z`.

This ordering must remain identical during inference.


## 1. Environment and dataset setup


### Google Drive


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

### Dataset archive check


In [ ]:
import os

ZIP_PATH = "/content/drive/MyDrive/Sign_language_new/Dataset_Merged.zip"

print("Exists:", os.path.exists(ZIP_PATH))

if os.path.exists(ZIP_PATH):
    size_gb = os.path.getsize(ZIP_PATH) / (1024**3)
    print(f"Size: {size_gb:.2f} GB")

### Extract dataset


In [ ]:
import zipfile
import os

EXTRACT_PATH = "/content/Dataset_Merged"

print("Extracting dataset...")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall("/content")

print("✅ Extraction complete")
print("Exists:", os.path.exists(EXTRACT_PATH))

### Inspect extracted folders


In [ ]:
import os

print("Folders in /content:")
print("-" * 60)

for item in os.listdir("/content"):
    path = os.path.join("/content", item)

    if os.path.isdir(path):
        print(item)

### Dataset root


In [ ]:
EXTRACT_PATH = "/content"

print("Dataset path:", EXTRACT_PATH)

### Dataset verification


In [ ]:
import os

IMAGE_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
)

classes = sorted([
    d for d in os.listdir(EXTRACT_PATH)
    if os.path.isdir(os.path.join(EXTRACT_PATH, d))
    and any(
        f.lower().endswith(IMAGE_EXTENSIONS)
        for f in os.listdir(os.path.join(EXTRACT_PATH, d))
    )
])

print("=" * 70)
print("DATASET VERIFICATION")
print("=" * 70)

total_images = 0

for cls in classes:

    folder = os.path.join(EXTRACT_PATH, cls)

    count = sum(
        1
        for f in os.listdir(folder)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    )

    total_images += count

    print(f"{cls:<15}: {count}")

print("-" * 70)

print("Classes:", len(classes))
print("Images :", total_images)

## 2. Dataset configuration


In [ ]:
DATASET_DIR = "/content"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

print("Dataset:", DATASET_DIR)

### Class discovery


In [ ]:
import os

IMAGE_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
)

classes = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
    and any(
        f.lower().endswith(IMAGE_EXTENSIONS)
        for f in os.listdir(os.path.join(DATASET_DIR, d))
    )
])

print("Number of classes:", len(classes))
print(classes)

### Train / validation / holdout datasets


In [ ]:
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=classes,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

holdout_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=classes,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Holdout batches:", tf.data.experimental.cardinality(holdout_ds).numpy())

### Validation and test split


In [ ]:
holdout_batches = tf.data.experimental.cardinality(
    holdout_ds
).numpy()

val_batches = holdout_batches // 2

val_ds = holdout_ds.take(val_batches)
test_ds = holdout_ds.skip(val_batches)

print("Validation batches:",
      tf.data.experimental.cardinality(val_ds).numpy())

print("Test batches:",
      tf.data.experimental.cardinality(test_ds).numpy())

### Dataset performance


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 3. Data augmentation


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomTranslation(
        height_factor=0.10,
        width_factor=0.10
    ),
    tf.keras.layers.RandomContrast(0.15),
], name="data_augmentation")

## 4. MobileNetV2 transfer-learning architecture


In [ ]:
from tensorflow.keras import layers, models

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = layers.Input(
    shape=(224, 224, 3)
)

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dense(
    256,
    activation="relu"
)(x)

x = layers.Dropout(0.40)(x)

outputs = layers.Dense(
    41,
    activation="softmax"
)(x)

model = models.Model(
    inputs,
    outputs
)

model.summary()

### Stage 1 compilation


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

### Training callbacks


In [ ]:
callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        "/content/sign_language_mobilenetv2_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        mode="max",
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

### Stage 1 training


In [ ]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

## 5. Fine-tuning


In [ ]:
# ============================================================
# STAGE 2 — FINE TUNING
# ============================================================

# Start by keeping the first ~100 layers frozen
base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

for layer in base_model.layers[100:]:
    layer.trainable = True

print("Total MobileNetV2 layers:", len(base_model.layers))
print(
    "Trainable MobileNetV2 layers:",
    sum(layer.trainable for layer in base_model.layers)
)

### Fine-tuning compilation


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

### Fine-tuning callbacks


In [ ]:
fine_tune_callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        "/content/sign_language_mobilenetv2_finetuned_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        mode="max",
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

### Fine-tuning training


In [ ]:
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=fine_tune_callbacks
)

## 6. Final test evaluation


In [1]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

import tensorflow as tf

# Load the best fine-tuned checkpoint
best_model = tf.keras.models.load_model(
    "/content/sign_language_mobilenetv2_finetuned_best.keras"
)

test_loss, test_accuracy = best_model.evaluate(
    test_ds,
    verbose=1
)

print("=" * 60)
print("FINAL TEST RESULT")
print("=" * 60)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4%}")

ValueError: File not found: filepath=/content/sign_language_mobilenetv2_finetuned_best.keras. Please ensure the file is an accessible `.keras` zip file.

### Classification report


In [ ]:
from sklearn.metrics import classification_report
import numpy as np

y_true = []
y_pred = []

for images, labels in test_ds:

    predictions = best_model.predict(
        images,
        verbose=0
    )

    y_true.extend(
        np.argmax(labels.numpy(), axis=1)
    )

    y_pred.extend(
        np.argmax(predictions, axis=1)
    )

# Make sure class names are strings
class_names = [str(c) for c in classes]

print("=" * 80)
print("CLASSIFICATION REPORT — ALL 41 CLASSES")
print("=" * 80)

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(41)),
    target_names=class_names,
    digits=4,
    zero_division=0
)

print(report)

### Confusion matrix


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(classes)))
)

plt.figure(figsize=(18, 16))

plt.imshow(cm)

plt.title("41-Class Sign Language Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(
    range(len(classes)),
    classes,
    rotation=90
)

plt.yticks(
    range(len(classes)),
    classes
)

plt.colorbar()

plt.tight_layout()
plt.show()

## 7. Save the trained model


In [ ]:
MODEL_PATH = "/content/sign_language_final_41class.keras"

best_model.save(MODEL_PATH)

print("✅ Model saved successfully")
print(MODEL_PATH)

### Save class mapping


In [ ]:
import json

CLASS_NAMES_PATH = "/content/sign_language_classes_41.json"

with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        classes,
        f,
        indent=4
    )

print("✅ Class names saved")
print(CLASS_NAMES_PATH)

### Verify saved artifacts


In [ ]:
import os

print("=" * 60)
print("SAVED FILES")
print("=" * 60)

for path in [MODEL_PATH, CLASS_NAMES_PATH]:

    print("\n", path)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        print(
            "Size:",
            round(os.path.getsize(path) / (1024 * 1024), 2),
            "MB"
        )

## 8. Local inference validation


In [1]:
import os
MODEL_PATH = r"C:\Users\franc\Downloads\sign_language_final_41class.keras"
CLASS_NAMES_PATH = r"C:\Users\franc\Downloads\sign_language_classes_41.json"

print("Model exists :", os.path.exists(MODEL_PATH))
print("Classes exists:", os.path.exists(CLASS_NAMES_PATH))

Model exists : True
Classes exists: True


### Load saved model and class mapping


In [6]:
import tensorflow as tf
import json
import numpy as np

model = tf.keras.models.load_model(MODEL_PATH)

with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as f:
    class_names = json.load(f)

print("=" * 60)
print("MODEL LOADED")
print("=" * 60)

print("Number of classes:", len(class_names))
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

print("\nClasses:")
for i, name in enumerate(class_names):
    print(f"{i:2d} -> {name}")

MODEL LOADED
Number of classes: 41
Input shape: (None, 224, 224, 3)
Output shape: (None, 41)

Classes:
 0 -> 1
 1 -> 10
 2 -> 2
 3 -> 3
 4 -> 4
 5 -> 5
 6 -> 6
 7 -> 7
 8 -> 8
 9 -> 9
10 -> A
11 -> B
12 -> C
13 -> D
14 -> E
15 -> F
16 -> G
17 -> H
18 -> HELLO
19 -> I
20 -> I_LOVE_YOU
21 -> J
22 -> K
23 -> L
24 -> M
25 -> N
26 -> NO
27 -> O
28 -> P
29 -> Q
30 -> R
31 -> S
32 -> T
33 -> THANK_YOU
34 -> U
35 -> V
36 -> W
37 -> X
38 -> Y
39 -> YES
40 -> Z


### Select a test image


In [13]:
import glob
import random

image_files = glob.glob(
    r"C:\Users\franc\Downloads\Dataset_Merged\**\*.jpg",
    recursive=True
)

print("Images found:", len(image_files))

TEST_IMAGE = random.choice(image_files)

print("Testing:", TEST_IMAGE)

Images found: 18450
Testing: C:\Users\franc\Downloads\Dataset_Merged\X\Dataset_new_X_0199.jpg


### Single-image prediction


In [14]:
import cv2
import numpy as np
IMG_SIZE = (224, 224)

image = cv2.imread(TEST_IMAGE)

if image is None:
    raise FileNotFoundError(TEST_IMAGE)

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

resized = cv2.resize(
    image_rgb,
    IMG_SIZE
)

input_image = resized.astype(np.float32)

input_image = np.expand_dims(
    input_image,
    axis=0
)

predictions = model.predict(
    input_image,
    verbose=0
)[0]

predicted_index = np.argmax(predictions)
predicted_class = class_names[predicted_index]
confidence = predictions[predicted_index]

print("=" * 60)
print("SINGLE IMAGE PREDICTION")
print("=" * 60)
print("Image     :", TEST_IMAGE)
print("Prediction:", predicted_class)
print(f"Confidence: {confidence:.2%}")

SINGLE IMAGE PREDICTION
Image     : C:\Users\franc\Downloads\Dataset_Merged\X\Dataset_new_X_0199.jpg
Prediction: X
Confidence: 96.76%


### Top-5 predictions


In [15]:
top5 = np.argsort(predictions)[-5:][::-1]

print("\nTOP 5 PREDICTIONS")
print("-" * 40)

for index in top5:
    print(
        f"{class_names[index]:<15} "
        f"{predictions[index]:.2%}"
    )


TOP 5 PREDICTIONS
----------------------------------------
X               96.76%
W               2.52%
V               0.72%
P               0.00%
Y               0.00%


## 9. Inference preprocessing note

The training architecture includes `tf.keras.applications.mobilenet_v2.preprocess_input()` **inside the saved model before the MobileNetV2 backbone**. The notebook's model summary shows the preprocessing operations between data augmentation and MobileNetV2. fileciteturn48file0L279-L304

Therefore, during deployment, the image should be supplied as **raw RGB pixel values in the 0–255 range** and the saved model should perform its own preprocessing.

This is important because applying `preprocess_input()` a second time during inference would preprocess the image twice.


In [12]:
# Correct deployment-side preprocessing for the saved model
# The saved model already contains MobileNetV2 preprocessing.
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
resized = cv2.resize(image_rgb, (224, 224))
input_image = np.expand_dims(resized.astype(np.float32), axis=0)

predictions = model.predict(input_image, verbose=0)[0]
predicted_index = int(np.argmax(predictions))
predicted_class = class_names[predicted_index]

print("Prediction:", predicted_class)
print(f"Confidence: {predictions[predicted_index]:.2%}")


Prediction: D
Confidence: 97.64%


In [16]:
import cv2
import numpy as np
import time

IMG_SIZE = (224, 224)

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("❌ Could not open webcam")

print("✅ Webcam opened")
print("Press Q to quit")

prev_time = time.time()

while True:

    ret, frame = cap.read()

    if not ret:
        print("❌ Failed to read frame")
        break

    # Mirror the webcam
    frame = cv2.flip(frame, 1)

    # --------------------------------------------------
    # PREPROCESS
    # --------------------------------------------------

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    resized = cv2.resize(
        rgb,
        IMG_SIZE
    )

    input_image = resized.astype(
        np.float32
    )

    input_image = np.expand_dims(
        input_image,
        axis=0
    )

    # --------------------------------------------------
    # PREDICTION
    # --------------------------------------------------

    predictions = model.predict(
        input_image,
        verbose=0
    )[0]

    predicted_index = np.argmax(predictions)

    predicted_class = class_names[
        predicted_index
    ]

    confidence = predictions[
        predicted_index
    ]

    # --------------------------------------------------
    # FPS
    # --------------------------------------------------

    current_time = time.time()

    fps = 1 / (
        current_time - prev_time
    )

    prev_time = current_time

    # --------------------------------------------------
    # DISPLAY
    # --------------------------------------------------

    text = f"{predicted_class} ({confidence:.1%})"

    cv2.putText(
        frame,
        text,
        (20, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0, 255, 0),
        3
    )

    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (20, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.imshow(
        "Sign Language Recognition",
        frame
    )

    # Q = quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()

✅ Webcam opened
Press Q to quit


# 10. Tools and techniques used

| Tool / technique | Role in the project |
|---|---|
| **Python** | Core programming language |
| **Google Colab** | Training environment and GPU execution |
| **TensorFlow / Keras** | Dataset loading, model building, training and evaluation |
| **MobileNetV2** | Pretrained CNN backbone used for transfer learning |
| **ImageNet pretrained weights** | Starting feature representations for transfer learning |
| **Data augmentation** | Random rotation, zoom, translation and contrast changes to improve robustness |
| **Adam optimizer** | Model optimization |
| **EarlyStopping** | Stops training when validation performance stops improving |
| **ModelCheckpoint** | Saves the best validation model |
| **ReduceLROnPlateau** | Reduces learning rate when validation loss stops improving |
| **scikit-learn** | Classification report, confusion matrix and dataset splitting |
| **OpenCV** | Image loading, resizing, RGB conversion and webcam/deployment processing |
| **MediaPipe Hand Landmarker** | Hand detection/landmark localization for the webcam deployment stage |
| **JSON** | Stores the 41-class mapping used during inference |

The notebook specifically uses augmentation with rotation, zoom, translation and contrast changes. fileciteturn48file0L231-L240 It uses MobileNetV2 with ImageNet weights, initially frozen and then partially fine-tuned. fileciteturn48file0L380-L424


# 11. Model training strategy

### Stage 1 — Transfer learning

The MobileNetV2 convolutional backbone is initially frozen. Only the classification head is trained. The head consists of:

- Global Average Pooling
- Batch Normalization
- Dense layer with 256 units
- Dropout of 0.40
- 41-class Softmax output

The model has approximately **2.60 million parameters**, with approximately **341k trainable parameters** during the initial transfer-learning stage. fileciteturn48file0L279-L372

### Stage 2 — Fine-tuning

The backbone is then unfrozen selectively: the first 100 MobileNetV2 layers remain frozen while later layers are made trainable. Fine-tuning uses a smaller learning rate of **1e-5**. fileciteturn48file0L583-L600 fileciteturn48file0L611-L617

This two-stage approach allows the classifier to first learn the sign-specific classification head and then adapt higher-level MobileNetV2 features to the sign-language dataset.


## Limitations
- M, N, S and T can be confused.
- Webcam conditions differ from training images.
- Background and lighting changes can affect predictions.
- 224 × 224 resizing can lose fine finger details.
- Single-frame prediction does not capture movement.
- Poor hand detection/cropping reduces accuracy.
- High confidence does not always mean correct prediction.

## Further Improvements
- Collect more real webcam images.
- Add different backgrounds, lighting and camera distances.
- Add more samples for M/N/S/T.
- Use MediaPipe for consistent hand localization/cropping.
- Compare with a custom CNN using stride = 1.
- Add an Unknown/Uncertain decision.
- Use LSTM, GRU or Transformer for dynamic gestures.
- Compare EfficientNet or MobileNetV3.
- Calibrate confidence and analyze errors class-by-class.